# 银行客户数据分析

使用 PySpark 对银行客户数据进行探索和分析。

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("bank_analysis").getOrCreate()

# 加载数据文件（分号分隔，带表头）
filePath = "/bank/bank-full.csv"
bankText = spark.read.option("header", "true").option("sep", ";").option("inferSchema", "true").csv(filePath)

bankText.cache()
bankText.show()

26/05/13 14:11:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
[Stage 2:=============================>                             (1 + 1) / 2]

+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+-------------+---------+-----------+---+
|age|        job| marital|          education|default|housing|loan|  contact|month|day_of_week|duration|campaign|pdays|previous|   poutcome|emp.var.rate|cons.price.idx|cons.conf.idx|euribor3m|nr.employed|  y|
+---+-----------+--------+-------------------+-------+-------+----+---------+-----+-----------+--------+--------+-----+--------+-----------+------------+--------------+-------------+---------+-----------+---+
| 56|  housemaid| married|           basic.4y|     no|     no|  no|telephone|  may|        mon|     261|       1|  999|       0|nonexistent|         1.1|        93.994|        -36.4|    4.857|     5191.0| no|
| 57|   services| married|        high.school|unknown|     no|  no|telephone|  may|        mon|     149|       1|  999|       0|nonexistent|         1.1|        93.

## 阶段三：数据探索与分析

In [2]:
# 1. 数据提炼：只保留 5 个字段
bankDF = bankText.select('age', 'job', 'marital', 'education', 'duration')
bankDF.show(5)

[Stage 4:>                                                          (0 + 1) / 1]

+---+---------+-------+-----------+--------+
|age|      job|marital|  education|duration|
+---+---------+-------+-----------+--------+
| 56|housemaid|married|   basic.4y|     261|
| 57| services|married|high.school|     149|
| 37| services|married|high.school|     226|
| 40|   admin.|married|   basic.6y|     151|
| 56| services|married|high.school|     307|
+---+---------+-------+-----------+--------+
only showing top 5 rows



In [3]:
# 2. 查看记录总数
print("总记录数:", bankDF.count())

[Stage 5:=============================>                             (1 + 1) / 2]

总记录数: 41188


In [4]:
# 3. 查看数据模式
bankDF.printSchema()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- duration: integer (nullable = true)



In [5]:
# 4. 注册临时视图
bankDF.createOrReplaceTempView("bank_tb")

In [6]:
# 5. 查看年龄小于 30 岁的客户信息
spark.sql("SELECT * FROM bank_tb WHERE age < 30").show()

[Stage 8:>                                                          (0 + 1) / 1]

+---+-----------+--------+-------------------+--------+
|age|        job| marital|          education|duration|
+---+-----------+--------+-------------------+--------+
| 24| technician|  single|professional.course|     380|
| 25|   services|  single|        high.school|      50|
| 25|   services|  single|        high.school|     222|
| 29|blue-collar|  single|        high.school|     137|
| 25| technician|  single|  university.degree|     174|
| 24| management|  single|  university.degree|     165|
| 28|   services| married|        high.school|      81|
| 28|   services| married|            unknown|     408|
| 27|blue-collar|  single|           basic.6y|     119|
| 27|blue-collar|  single|           basic.6y|     350|
| 27|     admin.|  single|  university.degree|     106|
| 26|     admin.|  single|  university.degree|     231|
| 29|   services|  single|        high.school|     425|
| 22|blue-collar|  single|           basic.9y|     195|
| 28|    unknown|  single|            unknown|  

In [7]:
# 6. 查看不同年龄段（<30）的客户人数
spark.sql("""
    SELECT age, COUNT(age) AS total_ages
    FROM bank_tb
    WHERE age < 30
    GROUP BY age
    ORDER BY age
""").show()

[Stage 9:>                                                          (0 + 2) / 2]

+---+----------+
|age|total_ages|
+---+----------+
| 17|         5|
| 18|        28|
| 19|        42|
| 20|        65|
| 21|       102|
| 22|       137|
| 23|       226|
| 24|       463|
| 25|       598|
| 26|       698|
| 27|       851|
| 28|      1001|
| 29|      1453|
+---+----------+



In [8]:
# 7. 查看有几种婚姻状况
bankDF.select("marital").distinct().show()

[Stage 12:=============================>                            (1 + 1) / 2]

+--------+
| marital|
+--------+
| unknown|
|divorced|
| married|
|  single|
+--------+



## 阶段四：自行练习

In [9]:
# 练习1：查询每种婚姻状况对应的人数
spark.sql("""
    SELECT marital, COUNT(*) AS total_people
    FROM bank_tb
    GROUP BY marital
""").show()

+--------+------------+
| marital|total_people|
+--------+------------+
| unknown|          80|
|divorced|        4612|
| married|       24928|
|  single|       11568|
+--------+------------+



In [10]:
# 练习2：查询不同年龄段客户的平均通话时长
spark.sql("""
    SELECT age, AVG(duration) AS avg_duration
    FROM bank_tb
    GROUP BY age
    ORDER BY age
""").show(100)

+---+------------------+
|age|      avg_duration|
+---+------------------+
| 17|             420.0|
| 18| 321.7857142857143|
| 19|             271.5|
| 20| 288.4923076923077|
| 21| 264.2450980392157|
| 22|250.92700729927006|
| 23|281.27433628318585|
| 24| 282.9049676025918|
| 25|259.98327759197326|
| 26| 263.5300859598854|
| 27|269.46650998824913|
| 28|270.02097902097904|
| 29| 258.2436338609773|
| 30|255.10326721120185|
| 31|257.31381612737545|
| 32|254.69772481040087|
| 33| 259.7075831969449|
| 34|255.81146131805158|
| 35| 256.2296759522456|
| 36|251.30898876404495|
| 37| 256.3857627118644|
| 38| 263.0575692963753|
| 39|254.71298882681563|
| 40|247.89147286821705|
| 41|254.45070422535213|
| 42|253.01663747810858|
| 43|277.76018957345974|
| 44|250.28684470820968|
| 45|266.31006346328195|
| 46|258.36990291262134|
| 47| 248.3739224137931|
| 48|251.27272727272728|
| 49|239.87604290822406|
| 50| 262.4114285714286|
| 51| 243.1657824933687|
| 52|257.17971758664953|
| 53| 259.7830832196453|


In [11]:
spark.stop()